# 05. 내보내기(Export) & 배포

학습한 모델을 파이썬 밖에서도 쓸 수 있게 **다른 포맷으로 변환**합니다.

배우는 것
1. 왜 내보내는가 / 어떤 포맷이 있나
2. ONNX로 내보내기
3. 내보낸 모델로 추론 (Ultralytics 경유)
4. 포맷 선택 가이드

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import torch
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
model = YOLO("yolo26n.pt")

## 1. 왜 내보내나?

`.pt`는 PyTorch/Ultralytics 환경이 있어야 돌아갑니다. 실제 배포 환경(모바일, 웹, C++ 서버, 엣지 디바이스)은 PyTorch가 없거나 무거워서 못 씁니다. 그래서 **가볍고 이식성 좋은 포맷**으로 바꿔요.

| 포맷 | 용도 |
|---|---|
| **ONNX** | 범용 표준. 대부분의 런타임/언어에서 실행 (가장 무난) |
| **TorchScript** | PyTorch C++ 배포 |
| **CoreML** | Apple 기기 (iOS/macOS) — 이 맥에 적합 |
| **TFLite** | 안드로이드/모바일/엣지 |
| **TensorRT** | NVIDIA GPU 초고속 추론 (이 맥은 불가) |
| **OpenVINO** | Intel CPU 최적화 |

## 2. ONNX로 내보내기

가장 범용적인 ONNX로 변환합니다. `.onnx` 파일이 원본 `.pt` 옆에 생성돼요.

In [ ]:
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    opset=12,        # ONNX 연산자 버전
    simplify=True,   # 그래프 단순화
    dynamic=False,   # 입력 크기 고정
)
print("내보낸 파일:", onnx_path)

## 3. 내보낸 모델로 추론하기

Ultralytics는 내보낸 파일도 `YOLO()`로 바로 불러 추론할 수 있습니다. (내부적으로 해당 런타임 사용)

In [ ]:
onnx_model = YOLO(onnx_path)
r = onnx_model.predict("https://ultralytics.com/images/bus.jpg", verbose=False)[0]
print("ONNX 모델 탐지 수:", len(r.boxes))
print("속도(ms):", r.speed)

from PIL import Image
Image.fromarray(r.plot()[..., ::-1])

## 4. Apple 기기용: CoreML (선택)

이 맥/아이폰에 배포한다면 CoreML이 최적입니다. (`coremltools` 필요할 수 있음)

In [ ]:
# 필요 시 주석 해제 (의존성 설치가 필요할 수 있음)
# coreml_path = model.export(format="coreml")
# print(coreml_path)
print("CoreML은 필요할 때 위 주석을 해제해 실행하세요.")

## 5. 포맷 선택 가이드

```
어디에 배포?
├─ 잘 모르겠다 / 범용            → ONNX
├─ iOS·macOS 앱                 → CoreML
├─ 안드로이드·모바일·엣지        → TFLite
├─ NVIDIA GPU 서버 (최고속도)   → TensorRT
├─ Intel CPU 서버               → OpenVINO
└─ PyTorch C++                  → TorchScript
```

> **팁**: 내보낸 뒤 반드시 `.pt` 결과와 탐지 개수·박스가 비슷한지 비교하세요. 양자화(`int8=True`)를 쓰면 빨라지지만 정확도가 떨어질 수 있어 검증이 필수입니다.

---
### 커리큘럼 완료 (탐지 코어)
00 개요 → 01 추론 → 02 데이터셋 → 03 학습 → 04 검증 → **05 배포** ✅

**보너스 →** `06_tracking` : 비디오 객체 추적 (탐지 응용)